In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# IBM Circuit function

> **Note:** * Qiskit Functionsは、IBM Quantum&reg; Premium Plan、Flex Plan、およびOn-Prem（IBM Quantum Platform API経由）Planのユーザーのみが利用できる実験的な機能です。プレビューリリースの状態であり、変更される可能性があります。

## 概要
IBM&reg; Circuit functionは、[抽象PUB](./primitive-input-output)を入力として受け取り、緩和された期待値を出力として返します。このCircuit functionには、研究者がアルゴリズムやアプリケーションの発見に集中できるよう、自動化されたカスタマイズパイプラインが含まれています。

## 説明
PUBを送信すると、抽象Circuitとオブザーバブルが自動的にトランスパイルされ、ハードウェア上で実行され、後処理によって緩和された期待値が返されます。これを実現するために、以下のツールを組み合わせています：

- [Qiskit Transpiler Service](./qiskit-transpiler-service)：AIによるトランスパイルパスとヒューリスティックなトランスパイルパスの自動選択により、抽象Circuitをハードウェア最適化されたISA Circuitに変換します
- [ユーティリティスケールの計算に必要なエラー抑制と緩和](./error-mitigation-and-suppression-techniques)：測定とゲートのtwirling、ダイナミカルデカップリング、Twirled Readout Error eXtinction（TREX）、Zero-Noise Extrapolation（ZNE）、Probabilistic Error Amplification（PEA）を含みます
- [Qiskit Runtime Estimator](./get-started-with-primitives)：ISA PUBをハードウェア上で実行し、緩和された期待値を返します

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## はじめに
[APIキー](http://quantum.cloud.ibm.com/)を使って認証を行い、以下のようにQiskit Functionを選択します。（このスニペットは、すでに[アカウントを保存済み](/guides/functions#install-qiskit-functions-catalog-client)であることを前提としています。）

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## 例
まず、この基本的な例を試してみてください：

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

Qiskit Functionワークロードの[ステータス](/guides/functions#check-job-status)を確認したり、以下のように[結果](/guides/functions#retrieve-results)を取得したりできます：

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


結果は[Estimatorの結果](/guides/primitive-input-output#estimator-output)と同じ形式です：

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## 入力
IBM Circuit functionが受け付けるすべての入力パラメーターについては、以下の表を参照してください。続く[オプション](#options)セクションでは、利用可能な`options`について詳しく説明します。

| 名前      | 型                         | 説明                                                                                                                                                                                                                                | 必須 | デフォルト                                                               | 例                                       |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | クエリを実行するBackendの名前。                                                                                                                                                                                                      | はい | なし                                                                     | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | `(circuit, observables)` または `(circuit, observables, parameter_values)` のようなタプルなど、抽象PUBライク（primitive unified bloc）オブジェクトのイテラブル。詳しくは[PUBの概要](/guides/primitive-input-output#overview-of-pubs)を参照してください。Circuitは抽象（非ISA）でも構いません。 | はい | なし                                                                     | (circuit, observables, parameter_values) |
| options   | dict                       | 入力オプション。詳細は**オプション**セクションを参照してください。                                                                                                                                                                  | いいえ | 詳細は**オプション**セクションを参照してください。                        | `{"optimization_level": 3}`               |
| instance  | str                        | 使用するインスタンスのクラウドリソース名（その形式で指定）。                                                                                                                                                                        | いいえ | アカウントが複数のインスタンスにアクセスできる場合、ランダムに1つ選択されます。 | `CRN`                                    |

### オプション
#### 構造
Qiskit Runtimeプリミティブと同様に、IBM Circuit functionのオプションはネストされた辞書として指定できます。``optimization_level`` や ``mitigation_level`` などのよく使われるオプションは第1レベルにあります。その他のより高度なオプションは、``resilience`` などのカテゴリにまとめられています。

#### デフォルト値
オプションの値を指定しない場合、サービスによって定義されたデフォルト値が使用されます。

#### 緩和レベル
IBM Circuit functionは`mitigation_level`もサポートしています。緩和レベルは、ジョブに適用するエラー抑制および緩和の量を指定します。レベルが高いほど精度の高い結果が得られますが、処理時間が長くなります。エラー削減の程度は適用される手法によって異なります。緩和レベルは、エラー緩和と抑制の手法の詳細な選択を抽象化し、ユーザーがアプリケーションに適したコストと精度のトレードオフを判断できるようにします。以下の表は、各レベルに対応する手法を示しています。

> **Note:** 名前は似ていますが、`mitigation_level`はEstimatorの`resilience_level`とは異なるテクニックを適用します。

Qiskit Runtime Estimatorの``resilience_level``と同様に、``mitigation_level``はキュレートされたオプションの基本セットを指定します。緩和レベルに加えて手動で指定したオプションは、緩和レベルによって定義された基本オプションセットの上に適用されます。そのため、原則として、緩和レベルを1に設定しながら測定緩和をオフにすることも可能ですが、これは推奨されません。

| **緩和レベル** | **テクニック** |
|:-:|:-:|
| 1 [デフォルト] | ダイナミカルデカップリング + 測定twirling + TREX  |
| 2 | レベル1 + ゲートtwirling + ゲートフォールディングによるZNE |
| 3 | レベル1 + ゲートtwirling + PEAによるZNE |

以下の例は、緩和レベルの設定方法を示しています：

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### 利用可能なすべてのオプション
`mitigation_level`に加えて、IBM Circuit functionはコストと精度のトレードオフを細かく調整できる高度なオプションも提供しています。以下の表はすべての利用可能なオプションを示しています：

| オプション | サブオプション | サブサブオプション | 説明 | 選択肢 | デフォルト |
|-|-|-|-|-|-|
| default_precision |  |  | 精度を指定しないPUBまたは`run()`呼び出しに使用するデフォルトの精度。<br/>各入力PUBは独自の精度を指定できます。`run()`メソッドに精度が指定された場合、その値は独自の精度を指定していない`run()`呼び出し内のすべてのPUBに使用されます。  | float > 0 | 0.015625 |
| max_execution_time |  |  | 秒単位の最大実行時間。QPU使用量（ウォールクロック時間ではなく）に基づきます。QPU使用量は、QPUがジョブの処理に専念する時間です。ジョブがこの時間制限を超えた場合、強制的にキャンセルされます。 | 秒単位の整数、範囲 [1, 10800] |  |
| mitigation_level |  |  | 適用するエラー抑制と緩和の量。各レベルで使用される手法の詳細については、[緩和レベル](#mitigation-level)セクションを参照してください。 | 1 / 2 / 3 | 1 |
| optimization_level |  |  | Circuitに対して実行する最適化の量。[高いレベル](/guides/set-optimization)ほど最適化されたCircuitが生成されますが、トランスパイル時間が長くなります。 | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | ダイナミカルデカップリングを有効にするかどうか。手法の説明については[エラー抑制と緩和テクニック](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling)を参照してください。  | True/False | True |
|  | sequence_type |  | 使用するダイナミカルデカップリングシーケンス。<br/>* `XX`：シーケンス `tau/2 - (+X) - tau - (+X) - tau/2` を使用<br/>* `XpXm`：シーケンス `tau/2 - (+X) - tau - (-X) - tau/2` を使用<br/>* `XY4`：シーケンス<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` を使用 | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | 2-Qubit Cliffordゲートのtwirlingを適用するかどうか。 | True/False | False |
|  | enable_measure |  | 測定のtwirlingを有効にするかどうか。 | True/False | True |
| resilience | measure_mitigation |  | TREXの測定エラー緩和手法を有効にするかどうか。手法の説明については[エラー抑制と緩和テクニック](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex)を参照してください。  | True/False | True |
|  | zne_mitigation |  | Zero Noise Extrapolationエラー緩和手法をオンにするかどうか。手法の説明については[エラー抑制と緩和テクニック](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne)を参照してください。  | True/False | False |
|  | zne | amplifier | ノイズを増幅するために使用するテクニック。以下のいずれか：<br/> - `gate_folding`（デフォルト）：ノイズを増幅するために2-QubitゲートフォールディングGateを使用します。ノイズ係数がゲートのサブセットのみを増幅する必要がある場合、これらのゲートはランダムに選択されます。<br/><br/> - `gate_folding_front`：ノイズを増幅するために2-Qubitゲートフォールディングを使用します。ノイズ係数がゲートのサブセットのみを増幅する必要がある場合、これらのゲートはトポロジー的に順序付けされたDAG Circuitの先頭から選択されます。<br/><br/> - `gate_folding_back`：ノイズを増幅するために2-Qubitゲートフォールディングを使用します。ノイズ係数がゲートのサブセットのみを増幅する必要がある場合、これらのゲートはトポロジー的に順序付けされたDAG Circuitの末尾から選択されます。<br/><br/> - `pea`：Probabilistic error amplification（PEA）と呼ばれるテクニックを使用してノイズを増幅します。手法の説明については[エラー抑制と緩和テクニック](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea)を参照してください。  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | ノイズ増幅に使用するノイズ係数。 | floatのリスト；各float >= 1 | PEAの場合は(1, 1.5, 2)、それ以外は(1, 3, 5)。 |
|  |  | extrapolator | フィット外挿モデルを評価するノイズ係数。このオプションは実行やモデルフィッティングには影響しません；`extrapolator`オブジェクトが評価されて`evs_extrapolated`と`stds_extrapolated`というデータフィールドに返される点のみを決定します。 | `exponential`、`linear`、`double_exponential`、`polynomial_degree_(1 <= k <= 7)` の1つ以上 | (`exponential`, `linear`) |
|  | pec_mitigation |  | Probabilistic Error Cancellationエラー緩和手法をオンにするかどうか。手法の説明については[エラー抑制と緩和テクニック](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec)を参照してください。  | True/False | False |
|  | pec | max_overhead | 許容される最大Circuit samplingオーバーヘッド。最大値なしの場合は`None`。 | None / 1より大きい整数 | 100 |

以下の例では、緩和レベルを1に設定すると最初はZNE緩和がオフになりますが、`zne_mitigation`を`True`に設定すると`mitigation_level`の関連設定が上書きされます。

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## 出力
Circuit functionの出力は[PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)であり、以下の2つのフィールドが含まれます：

- 1つ以上の[PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult)オブジェクト。`PrimitiveResult`から直接インデックスを指定してアクセスできます。

- ジョブレベルのメタデータ。

各`PubResult`には`data`フィールドと`metadata`フィールドが含まれます。

- `data`フィールドには、少なくとも期待値の配列（`PubResult.data.evs`）と標準誤差の配列（`PubResult.data.stds`）が含まれます。使用するオプションによっては、さらに多くのデータが含まれることもあります。

- `metadata`フィールドにはPUBレベルのメタデータ（`PubResult.metadata`）が含まれます。

以下のコードスニペットは`PrimitiveResult`（および関連する`PubResult`）の形式を説明しています。